# Project 4: Music Popularity Prediction


## Problem Statement

The objectives of this project are to:
1. **Predict** the popularity of a *new*, unreleased song.
2. Minimize the **cross-validated RMSE** for model accuracy.

#### Comment

Perfect mention of CV-RMSE! Just a minor tweak: make sure to avoid the word 'accuracy' here. In DS, accuracy is strictly a classification metric. For regression, we want to talk about minimizing error ( thus the RMSE ) or maximizing predictive performance.

## Methodology

* **Data**: Top 200 Weekly (Global) Charts of Spotify in 2020 & 2021
* **Source**: AWS, https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv
* **Methods**:
  * Data cleaning
  * Decision-tree (ML)

## Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import missingno as msno
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import plotly.express as px

## IDA

* Where is the data located, size, do I have access, etc.

In [ ]:
url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv"

In [ ]:
!curl -s -I {url}

In [ ]:
!curl -s -O {url}

In [ ]:
# Verify the data was loaded
!ls -la

In [ ]:
# Provides the header / column names
!head -1 Spotify.csv | tr , '\n' | cat -n

## EDA

In [ ]:
orig_dat = pd.read_csv( url, index_col = 0)
orig_dat.info()

In [ ]:
# helper functions

def metadata( dataframe ):
  '''Given a dataframe, returns a dataframe of metadata about the dataframe'''
  metadata_df = pd.DataFrame()
  metadata_df["Data_types"] = dataframe.dtypes
  metadata_df["Count"] = len(dataframe)
  metadata_df["Nulls"] = dataframe.isnull().sum()
  metadata_df["Nulls_pct"] = ( metadata_df["Nulls"] / metadata_df["Count"] * 100 ).round(1)
  metadata_df["Memory"] = dataframe.memory_usage( deep = True)
  metadata_df["NUnique"] = dataframe.nunique()
  metadata_df["NUnique_pct"] = (dataframe.nunique() / metadata_df["Count"] * 100).round(1)
  metadata_df = metadata_df.join( dataframe.describe( include = "all" ).transpose() )
  metadata_df = metadata_df.astype( { "count" : int } )
  if dataframe.select_dtypes(include=['number']).shape[1] :
    metadata_df["IRQ"] = metadata_df["75%"] - metadata_df["25%"]
    metadata_df["range"] = metadata_df["max"] - metadata_df["min"]
    metadata_df["sum"] = metadata_df["mean"] * metadata_df["count"]
    metadata_df = (
      metadata_df
      .rename( columns = {
        "25%" : "Q1_25%",
        "50%" : "Q2_median",
        "75%" : "Q3_75%",
        }
      )
    )
  return metadata_df

def cols_to_drop( dataframe ):
  '''Given a dataframe, returns columns that should likely be dropped'''
  md = metadata( dataframe )
  filter = md["Nulls_pct"] >= 40
  return md[ filter ]["Nulls_pct"].to_dict()

def likely_ids( dataframe ):
  '''Given a dataframe, returns a dictionary of likely ID columns'''
  md = metadata( dataframe )
  return md["NUnique_pct"][ md["NUnique_pct"] > 95 ].sort_values( ascending = False )

In [ ]:
metadata(orig_dat)

In [ ]:
cols_to_drop(orig_dat)

In [ ]:
likely_ids(orig_dat)

### Comment

Exploring the metadata - nice!

### Initial Observations
1. **Incorrect Data Types** - Several variables (`Artist Followers`, `Tempo`, `Streams`, etc) are the incorrect data type.
2. **Missing Data** - The most common data type for many variables (such as `Genre` and `Energy`) are empty but being picked up as "not null".
3. **Longitudinal Data** - `Weeks Charted` is a list with multiple dates, or timestamps, where the song was in the charts.
4. **Shape** - There are 1,556 rows and 22 columns.
5. **Unique Identifier** - There are 2 *actual* unique identifier called `Song ID` and `Song Name`.
6. **Messy String Data** - The variables `Chord` and `Genre` likely contain duplicates separated by delimeters such as the comma (,) and slash (/), among other possibilities not observed by the metadata table.

### Comment

Isn't it odd that Song ID is not a unique identifier.  What does that suggest?



### Incorrect Data Types

In [ ]:
dtypes_dat = orig_dat.copy()

In [ ]:
for col in dtypes_dat:
  print(f'"{col}" is type: * {dtypes_dat[col].dtype} *')

### Comment

Effectively the same as this ...



In [ ]:
metadata(orig_dat)["Data_types"].sort_values()

#### Int64

* Identify columns that should be `numeric`, but aren't currently being read correctly.
* This includes `int64`, which doesn't allow `null` values. I'm retaining `null` values to make a decision later.

In [ ]:
int64_vars = [
    'Highest Charting Position',
    'Number of Times Charted',
    'Streams',
    'Artist Followers', # has NA
    'Popularity', # has NA
    'Duration (ms)', # has NA
    'Tempo', # shown as a float64, but should be int (no musician uses decimals)
        #'Song ID', # randomized vals
]

for col in dtypes_dat[int64_vars]:
  print(f'"{col}" is type: * {dtypes_dat[col].dtype} *')

dtypes_dat[int64_vars]

* Use `regex` to test for incorrect values within expected `numeric` columns.

In [ ]:
non_num = dtypes_dat["Tempo"].str.contains(r'\D', regex=True)
dtypes_dat[non_num]

* After investigation, `Streams` won't coerce into `Int64` using standard methods due to the comma (,) value.
* `Tempo` also contains a period (.) followed by three numbers.
* We remove these false digits using `regex`.

In [ ]:
# remove commas in 'Streams' to coerce safely in a new column
dtypes_dat['streams_num'] = dtypes_dat['Streams'].str.replace(',', '').astype('Int64')
dtypes_dat['streams_num'].head(3)

In [ ]:
# remove the period and three numbers in 'Tempo'
dtypes_dat['tempo_num'] = dtypes_dat['Tempo'].str.extract(r'(\d{3})', expand=False).astype('Int64')
dtypes_dat['tempo_num'].head(3)

In [ ]:
# recall remaining cols
## int64_vars

# redefine
int64_vars = [
    'Highest Charting Position',
    'Number of Times Charted',
    'Artist Followers',
    'Popularity',
    'Duration (ms)'
]

# convert
for col in int64_vars:
    dtypes_dat[col + '_num'] = pd.to_numeric(dtypes_dat[col], errors='coerce').astype('Int64') # will rename later

In [ ]:
# validate
for col in dtypes_dat:
  print(f'"{col}" is type: * {dtypes_dat[col].dtype} *')

dtypes_dat.head(3)

#### Floats

In [ ]:
float_vars = [
    'Danceability',
    'Energy',
    'Loudness',
    'Speechiness',
    'Acousticness',
    'Liveness',
    'Valence',
]


for col in dtypes_dat[float_vars]:
  print(f'"{col}" is type: * {dtypes_dat[col].dtype} *')

dtypes_dat[float_vars]

In [ ]:
# convert
for col in float_vars:
    dtypes_dat[col + '_num'] = pd.to_numeric(dtypes_dat[col], errors='coerce').astype(float)

In [ ]:
# validate
for col in dtypes_dat:
  print(f'"{col}" is type: * {dtypes_dat[col].dtype} *')

dtypes_dat.head(3)

#### Dates

The decision:
* `Week of Highest Charting` - create 2 new columns parsed by the "--" delimeter and calculate number of days charted.
* `Release Date` - create date and calculate number of days since release.
* Original `date` fields will be dropped.

In [ ]:
# parse
dtypes_dat[['week_start', 'week_end']] = dtypes_dat['Week of Highest Charting'].str.split('--', expand=True)
dtypes_dat.head(3)

In [ ]:
# define date cols
date_vars = [
    'week_start',
    'week_end',
    'Release Date'
]

for col in date_vars:
    dtypes_dat[col + '_num'] = pd.to_datetime(dtypes_dat[col], errors='coerce')

In [ ]:
# validate
for col in dtypes_dat:
  print(f'"{col}" is type: * {dtypes_dat[col].dtype} *')

dtypes_dat.head(3)

In [ ]:
# define today
today = pd.Timestamp.today().normalize() # normalize to midnight

# calculate days
dtypes_dat['days_since_release'] = (today - dtypes_dat['Release Date_num']).dt.days

# ensure Int64
dtypes_dat['days_since_release'] = dtypes_dat['days_since_release'].astype('Int64')

# validate
dtypes_dat['days_since_release'].head()

In [ ]:
# calculate days charted
dtypes_dat['days_charted'] = (dtypes_dat['week_end_num'] - dtypes_dat['week_start_num']).dt.days

# ensure Int64
dtypes_dat['days_charted'] = dtypes_dat['days_charted'].astype('Int64')

# validate
dtypes_dat['days_charted'].head()

In [ ]:
dtypes_dat['days_charted'].nunique()

* All 7! Let's multiply by the amount of times charted and see how it changes.

In [ ]:
dtypes_dat['actual_days_charted'] = dtypes_dat['days_charted'] * dtypes_dat['Number of Times Charted']
dtypes_dat['actual_days_charted'].nunique()

In [ ]:
dtypes_dat['actual_days_charted'].head()

* Now we can use `actual_days_charted` as our **target variable**, since longer charting days can compound an artist's actual popularity by pushing the song on Spotify to more users.
* This should be a more granular metric than `Number of Times Charted`, but we'll check the correlations later to confirm this assumption.

#### Strings

* We are looking at `Chord` and `Genre` only.

In [ ]:
print(f'There are {dtypes_dat['Chord'].nunique()} unique chord values.')
print(f'There are {dtypes_dat['Genre'].nunique()} unique genre values.')

In [ ]:
dtypes_dat['Chord'].unique()

* We find all of the chords accounted for: A, A#/Bb, B, C, C#/Db, D, D#/Eb, E, F, F#/Gb, G, G#/Ab.
* We also find a false whitespace.
* We remove the whitespace and don't opt for one-hot encoding in this round.

In [ ]:
dtypes_dat['Genre'].unique()

* There is a great amount of genres that overlap. This could be its own predictor in and of itself.
* The goal is to separate `Genre` and treat each model as its own: for example, one model for "pop", another for "rock", another for "hip hop", and so forth.
  * My key assumption is that "pop" constantly dominates the charts thanks to the music industry.
  * For the purpose of this project, I'm opting to completely *drop* this column due to its nuance and complication from both a data science and musician's perspective.

In [ ]:
dtypes_dat['chord'] = dtypes_dat['Chord'].astype(str).str.strip()
dtypes_dat['chord'] = dtypes_dat['chord'].replace('', np.nan)
dtypes_dat['chord'].unique()

In [ ]:
# one-hot encode
dtypes_dat = pd.get_dummies(dtypes_dat, columns=["chord"], prefix="", prefix_sep="")
dtypes_dat

### Transforming

* I define a new, cleaner dataset to work with for my decision tree model.

In [ ]:
full_dat = dtypes_dat.copy()

In [ ]:
full_dat.columns

In [ ]:
# cols to keep
cols_to_keep = ['streams_num', 'tempo_num', 'Highest Charting Position_num',
       'Number of Times Charted_num', 'Artist Followers_num', 'Popularity_num',
       'Duration (ms)_num', 'Danceability_num', 'Energy_num', 'Loudness_num',
       'Speechiness_num', 'Acousticness_num', 'Liveness_num', 'Valence_num',
       'days_since_release', 'actual_days_charted', 'Artist', 'Song Name',
        'A', 'A#/Bb', 'B', 'C', 'C#/Db', 'D', 'D#/Eb', 'E', 'F', 'F#/Gb', 'G', 'G#/Ab']
full_dat = full_dat[cols_to_keep]

In [ ]:
# rename
full_dat = full_dat.rename(columns={
    'actual_days_charted': 'days_charted', # target var
    'days_since_release': 'days_since_release',
    'chord': 'chord',
    'streams_num': 'streams',
    'tempo_num': 'tempo',
    'Highest Charting Position_num': 'highest_position',
    'Number of Times Charted_num': 'times_charted',
    'Artist Followers_num': 'followers',
    'Popularity_num': 'popularity',
    'Duration (ms)_num': 'duration_ms',
    'Danceability_num': 'danceability',
    'Energy_num': 'energy',
    'Loudness_num': 'loudness',
    'Speechiness_num': 'speechiness',
    'Acousticness_num': 'acousticness',
    'Liveness_num': 'liveness',
    'Valence_num': 'valence',
    'Artist': 'artist', # for id purposes
    'Song Name': 'song_name', # for id purposes
    'A': 'A',
    'A#/Bb': 'A#/Bb',
    'B': 'B',
    'C': 'C',
    'C#/Db': 'C#/Db',
    'D': 'D',
    'D#/Eb': 'D#/Eb',
    'E': 'E',
    'F': 'F',
    'F#/Gb': 'F#/Gb',
    'G': 'G',
    'G#/Ab': 'G#/Ab'
})
full_dat

In [ ]:
for col in full_dat:
  print(f'"{col}" is type: * {full_dat[col].dtype} *')

### Missing Data

In [ ]:
miss_dat = full_dat.copy()

In [ ]:
msno.matrix(miss_dat, labels=True)

* Dropping all `null` rows...

In [ ]:
main_dat = miss_dat.dropna()
msno.matrix(main_dat, labels=True)

In [ ]:
metadata(main_dat)

In [ ]:
main_dat.info()

### Visualizations

#### Random Observations

* Which songs have a `Tempo` of 999???

In [ ]:
false_temp = ( main_dat['tempo'] == 999 )
main_dat[false_temp]

In [ ]:
tempo_sorted = main_dat.sort_values(by="tempo", ascending=False)

subset = tempo_sorted.head(10)

fig_false_temp = px.line(
    subset,
    x="song_name",
    y="tempo",
    title="Top 10 Songs with Impossible BPM"
)

fig_false_temp.update_layout(
    yaxis_title = "Song Name",
    xaxis_title = "Tempo"
)

fig_false_temp.update_xaxes(tickangle=45)

fig_false_temp.show()

* Which songs had a #1 spot in the charts?

In [ ]:
main_dat.columns

In [ ]:
main_dat['days_charted'].sort_values(ascending=False)

* What do the `Streams` and `Duration (ms)` variables look like when plotted?

In [ ]:
fig_streams_duration = px.line(
    main_dat,
    x="streams",
    y="duration_ms",
    title="Song Duration (ms) vs. Total Streams"
    )

fig_streams_duration.update_layout(
    yaxis_title = "Duration (ms)",
    xaxis_title = "Total Streams"
)

fig_streams_duration.show()

* **Action:** Create a new column called `Duration (min)` for more intuitive reading.

* What does the relationship between number of `streams` and the number of `followers` an artist has look like?

In [ ]:
fig_streams_followers = px.line(
    main_dat,
    x="streams",
    y="followers",
    title="Artist's Spotify Followers vs. Total Streams")

fig_streams_followers.update_layout(
    yaxis_title = "Followers",
    xaxis_title = "Total Streams"
)

fig_streams_followers.show()

#### Correlation Matrix

In [ ]:
# remove objects
vars = main_dat.drop(columns=["artist", "song_name"])

In [ ]:
# correlation plot
corr = vars.corr().round(2)
corr_fig = px.imshow(corr, color_continuous_scale='picnic', title="Possible Features: Correlation Matrix")
corr_fig.show()

* As expected, `times_charted` and `highest_position` are repetitive of `days_charted`. We remove those 2 columns.
* `loudness` and `acousticness` is also highly correlated with `energy`. I'm choosing to remove `energy` for interpretability and preference.

In [ ]:
# drop highly correlated features and target var
features = main_dat.drop(columns=["artist", "song_name", "energy", "highest_position", "times_charted", "days_charted", "popularity", "streams"])

In [ ]:
corr = features.corr().round(2)
corr_fig = px.imshow(corr, color_continuous_scale='picnic', title="Selected Features: Correlation Matrix")
corr_fig.show()

## First Model

In [ ]:
features

In [ ]:
X = features
y = main_dat['days_charted']

numLoops = 500

rms_error = np.zeros(numLoops)

for idx in range(0,numLoops):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)
  model = DecisionTreeRegressor(max_depth=3)         # Arbitrarily choosing max_depth of 3
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  rms_error[idx] = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CV RMSE: {rms_error.mean().round(2)*1000}")

In [ ]:
importances = model.feature_importances_
tree_importances = pd.Series( importances, index = X.columns )

plt.figure()
# tree_importances.plot.bar()
tree_importances.sort_values( ascending = False ).plot.bar()
plt.title("Feature importances")
plt.ylabel('Feature Importance Score') ;

In [ ]:
( tree_importances.sort_values( ascending = False ) * 100 ).cumsum()

In [ ]:
fig = px.histogram(main_dat, x="days_charted")
fig.show()

In [ ]:
main_dat.sort_values(by='days_charted', ascending=False)

## Revisions

Things not previously checked:
1. **Duplicates** - these can add weight to the decision tree. For example, `followers` shows duplicates since they refer to the artist, not the song.
  * **Solution**: Drop columns where strong duplicates occur.
2. **Distribution** - outliers are certainly included in this set, as evidenced earlier by the `tempo` chart. The target variable, `days_charted`, is also *extremely* right-skewed.
  * **Solution**: `log` transform where appropriate.

* We can confirm, however, that `popularity` is ***not*** the most reliable "success" metric, given that songs who charted the longest weren't even in the Top 10 most popular ranks.

#### Duplicates

In [ ]:
for col in features:
  duplicate_count = features[col].duplicated().sum()
  total_rows = 1485
  percent_dups = ( duplicate_count / total_rows ) * 100

  print(f"Duplicates in {col}: {duplicate_count}, or {percent_dups.round(2)}%")

* The chords are binary, so these are being ignored for now.
* Opting to drop columns with more than 70% dups...
* And another set of 60%+ ...

In [ ]:
features2 = features.drop(columns=['tempo', 'days_since_release'])
features3 = features2.drop(columns=['followers', 'danceability', 'liveness'])

#### Distribution

In [ ]:
fig = px.histogram(main_dat, x="tempo")
fig.show()

In [ ]:
import seaborn as sns

sns.pairplot(features3) ;

#### Log-Transformations

In [ ]:
main_dat['days_charted_log'] = np.log1p(main_dat['days_charted'])
main_dat

In [ ]:
fig = px.histogram(main_dat, x="days_charted_log")
fig.show()

## Second Model

In [ ]:
X = features2
y = main_dat['days_charted_log']

numLoops = 500

rms_error = np.zeros(numLoops)

for idx in range(0,numLoops):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)
  model = DecisionTreeRegressor(max_depth=3)         # Arbitrarily choosing max_depth of 3
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  rms_error[idx] = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CV RMSE: {rms_error.mean().round(2)*1000}")

* This time, we optimize the ideal depth using loops.

In [ ]:
max_depths = [1,2,3,4,5,6,7,8,9,10]
rms_depth = np.zeros(len(max_depths))
std_depth = np.zeros(len(max_depths))

numLoops = 500

for n, depth in enumerate(max_depths):
  rms_error = np.zeros(numLoops)

  for idx in range(0,numLoops):
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)
    model = DecisionTreeRegressor(max_depth=depth)
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    rms_error[idx] = np.sqrt(mean_squared_error(y_test, y_pred))

  rms_depth[n] = rms_error.mean()
  std_depth[n] = rms_error.std( ddof = 1 )

In [ ]:
# Plot result
plt.figure(figsize = (8,5))
plt.plot(max_depths, rms_depth)
plt.xlabel('Max Depth')
plt.ylabel('RMSE')
plt.xlim(0, 10.5)
plt.grid()

## Third Model

In [ ]:
X_3 = features3
y_3 = main_dat['days_charted_log']

numLoops_3 = 500

rms_error_3 = np.zeros(numLoops_3)

for idx_3 in range(0,numLoops_3):
  X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_3,y_3,test_size=0.2)
  model_3 = DecisionTreeRegressor(max_depth=2)         # Arbitrarily choosing max_depth of 3
  model_3.fit(X_train_3,y_train_3)
  y_pred_3 = model_3.predict(X_test_3)
  rms_error_3[idx_3] = np.sqrt(mean_squared_error(y_test_3, y_pred_3))

print(f"CV RMSE: {rms_error_3.mean().round(2)*1000}")

In [ ]:
importances_3 = model_3.feature_importances_
tree_importances_3 = pd.Series( importances_3, index = X_3.columns )

plt.figure()
# tree_importances.plot.bar()
tree_importances_3.sort_values( ascending = False ).plot.bar()
plt.title("Feature importances")
plt.ylabel('Feature Importance Score') ;

## Future Development

* Revise the helper functions and import from GitHub
* Feature transformation and optimization
* Different target variable